# 02 — E0_team baselines on the canonical command split

Two CatBoost runs on the team split (`splits/team_*_idx.npy`):

- **E0_team_full** — my feature spec: `drop=['resolution','ItemID','SellerID']`, cat columns auto-detected from `object` dtypes + `SellerID`.
- **E0_team_clean** — team feature spec: `drop=['id','ItemID','SellerID','name_rus','description','brand_name','text','resolution']`, cat columns = the leftover object dtypes (`CommercialTypeName4`).

CatBoost config — same as my E0_canon (`fintech_approaches/fintech_experiment.ipynb` cell 5). Disclaimer: this is intentionally NOT the team config (`iterations=700`, no early stopping) so that all team_runs experiments share one config and only the feature/embedding axis varies.

In [1]:
from pathlib import Path
import json
import sys
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'notebooks'))
import utils  # noqa: E402  pylint: disable=wrong-import-position

DATA_PATH = '/Users/sofya/Desktop/ВКР /data/ml_ozon_ounterfeit_train.csv'
SPLITS = ROOT / 'splits'
PROBA  = ROOT / 'proba'
RESULTS = ROOT / 'results'
PROBA.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH, encoding='utf-8')
train_idx = np.load(SPLITS / 'team_train_idx.npy')
val_idx   = np.load(SPLITS / 'team_val_idx.npy')
test_idx  = np.load(SPLITS / 'team_test_idx.npy')
y_test_saved = np.load(SPLITS / 'y_test.npy')

y_train = df.iloc[train_idx]['resolution'].astype('int8').values
y_val   = df.iloc[val_idx]['resolution'].astype('int8').values
y_test  = df.iloc[test_idx]['resolution'].astype('int8').values
assert (y_test == y_test_saved).all(), 'y_test drift between split notebook and now'
spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

CATBOOST_BASE = dict(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    eval_metric='AUC',
    scale_pos_weight=spw,
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100,
)
print(f'sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}')
print(f'scale_pos_weight (train) = {spw:.4f}')
print(f'CATBOOST_BASE = {CATBOOST_BASE}')

sizes: train=69453, val=69335, test=58410
scale_pos_weight (train) = 15.3959
CATBOOST_BASE = {'iterations': 1000, 'depth': 6, 'learning_rate': 0.05, 'eval_metric': 'AUC', 'scale_pos_weight': 15.395892351274787, 'early_stopping_rounds': 50, 'random_seed': 42, 'verbose': 100}


In [2]:
# ---- helpers shared by both runs
def prep_split(df_part, feature_cols, cat_cols):
    """Prepare X for CatBoost: fillna(0) for numeric, fillna('nan').astype(str) for cat."""
    X = df_part[feature_cols].copy()
    num_cols = [c for c in feature_cols if c not in cat_cols]
    if num_cols:
        X[num_cols] = X[num_cols].fillna(0)
    for c in cat_cols:
        X[c] = X[c].fillna('nan').astype(str)
    return X


def run_catboost(name, feature_cols, cat_cols, save_proba_to):
    print(f'\n========== {name} ==========')
    print(f'feature_cols (n={len(feature_cols)}): cat_cols={cat_cols}')
    X_tr = prep_split(df.iloc[train_idx], feature_cols, cat_cols)
    X_va = prep_split(df.iloc[val_idx],   feature_cols, cat_cols)
    X_te = prep_split(df.iloc[test_idx],  feature_cols, cat_cols)

    model = CatBoostClassifier(**CATBOOST_BASE)
    t0 = time.time()
    model.fit(X_tr, y_train, cat_features=cat_cols, eval_set=(X_va, y_val), use_best_model=True)
    dt = time.time() - t0
    proba = model.predict_proba(X_te)[:, 1]
    assert proba.shape == y_test.shape
    np.save(save_proba_to, proba.astype('float32'))
    metrics = utils.evaluate(y_test, proba, name)
    metrics.update({
        'best_iteration': int(model.best_iteration_),
        'converged': bool(model.best_iteration_ < CATBOOST_BASE['iterations'] - 1),
        'fit_seconds': round(dt, 1),
        'n_features': len(feature_cols),
        'cat_features': cat_cols,
    })
    print(f'best_iter={metrics["best_iteration"]}, fit={dt:.1f}s, proba saved -> {save_proba_to}')
    return metrics

In [3]:
# ---- E0_team_full: my feature spec on team split
feature_cols_full = [c for c in df.columns if c not in ['resolution', 'ItemID', 'SellerID']]
obj_cols = df[feature_cols_full].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols_full = list(obj_cols)  # SellerID was dropped from features, so no need to re-append it
metrics_full = run_catboost('E0_team_full', feature_cols_full, cat_cols_full, PROBA / 'test_proba_e0_team_full.npy')


========== E0_team_full ==========
feature_cols (n=42): cat_cols=['brand_name', 'description', 'name_rus', 'CommercialTypeName4']


0:	test: 0.9037602	best: 0.9037602 (0)	total: 109ms	remaining: 1m 49s


100:	test: 0.9494162	best: 0.9494544 (96)	total: 2.98s	remaining: 26.6s


200:	test: 0.9501590	best: 0.9502426 (155)	total: 5.99s	remaining: 23.8s


300:	test: 0.9515349	best: 0.9515349 (300)	total: 9.09s	remaining: 21.1s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9515348842
bestIteration = 300

Shrink model to first 301 iterations.


E0_team_full              | ROC-AUC=0.9366 | PR-AUC=0.6812 | R@P90=0.0691
best_iter=300, fit=11.2s, proba saved -> /Users/sofya/women-in-data-thesis/.claude/worktrees/dazzling-vaughan/team_runs/proba/test_proba_e0_team_full.npy


In [4]:
# ---- E0_team_clean: team feature spec on team split
team_drop = ['id', 'ItemID', 'SellerID', 'name_rus', 'description', 'brand_name', 'text', 'resolution']
feature_cols_clean = [c for c in df.columns if c not in team_drop]
cat_cols_clean = df[feature_cols_clean].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
print(f'expected: cat_cols_clean == ["CommercialTypeName4"], got: {cat_cols_clean}')
metrics_clean = run_catboost('E0_team_clean', feature_cols_clean, cat_cols_clean, PROBA / 'test_proba_e0_team_clean.npy')

expected: cat_cols_clean == ["CommercialTypeName4"], got: ['CommercialTypeName4']

========== E0_team_clean ==========
feature_cols (n=38): cat_cols=['CommercialTypeName4']


0:	test: 0.8988791	best: 0.8988791 (0)	total: 15.2ms	remaining: 15.2s


100:	test: 0.9270727	best: 0.9270920 (99)	total: 1.52s	remaining: 13.6s


200:	test: 0.9296641	best: 0.9296995 (199)	total: 3.05s	remaining: 12.1s


300:	test: 0.9310278	best: 0.9314758 (293)	total: 4.53s	remaining: 10.5s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9314758359
bestIteration = 293

Shrink model to first 294 iterations.
E0_team_clean             | ROC-AUC=0.9050 | PR-AUC=0.5301 | R@P90=0.0022
best_iter=293, fit=5.3s, proba saved -> /Users/sofya/women-in-data-thesis/.claude/worktrees/dazzling-vaughan/team_runs/proba/test_proba_e0_team_clean.npy


In [5]:
# ---- Persist a small json summary for downstream notebooks
summary = {
    'catboost_base': {k: v for k, v in CATBOOST_BASE.items() if k != 'verbose'},
    'E0_team_full':  metrics_full,
    'E0_team_clean': metrics_clean,
    'delta_full_minus_clean': {
        'r_at_p90': metrics_full['r_at_p90'] - metrics_clean['r_at_p90'],
        'pr_auc':   metrics_full['pr_auc']   - metrics_clean['pr_auc'],
        'roc_auc':  metrics_full['roc_auc']  - metrics_clean['roc_auc'],
    },
}
with open(RESULTS / 'e0_team_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(json.dumps(summary['delta_full_minus_clean'], indent=2))

{
  "r_at_p90": 0.06698255438294207,
  "pr_auc": 0.15110754107930613,
  "roc_auc": 0.03159414068843347
}
